# 12 — Investigation & Explainability

## Objective
Provide investigator-friendly explanations for high-scoring transactions:
- Feature contribution via a simple perturbation / z-score view
- Entity neighbourhood summary (same user / device / IP)
- Optional SHAP values if the library is available


In [ ]:

from pathlib import Path
import sys
import numpy as np
import pandas as pd
import joblib

ROOT = Path.cwd()
if not (ROOT / "data").exists(): ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

from data_utils import load_processed

X_test = load_processed("X_test")
model = joblib.load(ROOT / "models" / "final_model.joblib")
X_f = X_test.drop(columns=["class"])
scores = model.predict_anomaly_score(X_f)
X_test = X_test.copy()
X_test["anomaly_score"] = scores
X_test["anomaly_rank"] = X_test["anomaly_score"].rank(ascending=False).astype(int)

top = X_test.nsmallest(5, "anomaly_rank")
print("Top-5 anomalous transactions (feature snapshot):")
display(top.head())

# Simple explanation: which features deviate most from median
med = X_f.median()
mad = (X_f - med).abs().median().replace(0, 1e-9)
for idx, row in top.iterrows():
    z = ((row[X_f.columns] - med) / (1.4826 * mad)).abs().sort_values(ascending=False)
    print(f"\n--- Rank {int(row['anomaly_rank'])} | score={row['anomaly_score']:.3f} | label={row['class']} ---")
    print("Top deviant features:")
    print(z.head(8).round(2))
